In [ ]:
!pip install datasets transformers jiwer torchaudio --quiet

In [ ]:
import os
import pandas as pd
from datasets import Dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

In [ ]:
# DATA_DIR = "/kaggle/working/ASR_Nepali_0/asr_nepali"
DATA_DIR = "/kaggle/input/asr-nepali-openslr-final/asr_nepali/"
TSV_FILE = os.path.join(DATA_DIR, "utt_spk_text.tsv")

In [ ]:
df = pd.read_csv(TSV_FILE, sep="\t", header=None, names=["utt_id","spk_id","sentence"])

In [ ]:
# This is alternate

import os

# Step 1: Get all .flac files in the dataset once
all_flac_files = []
for root, dirs, files in os.walk(os.path.join(DATA_DIR, "data")):
    for f in files:
        if f.endswith(".flac"):
            all_flac_files.append(os.path.join(root, f))

# Step 2: Map utt_id to path
utt_to_path = {}
for f in all_flac_files:
    utt_id = os.path.basename(f).split(".")[0]  # assumes filename starts with utt_id
    utt_to_path[utt_id] = f

# Step 3: Assign paths to dataframe
df["path"] = df["utt_id"].map(utt_to_path)
df = df.dropna()


In [ ]:
# paths = []
# for utt in df.utt_id:
#     found = None
#     for root, dirs, files in os.walk(os.path.join(DATA_DIR,"data")):
#         for f in files:
#             if f.startswith(utt) and f.endswith(".flac"):
#                 found = os.path.join(root,f)
#                 break
#         if found:
#             break
#     paths.append(found)
# df["path"] = paths
# df = df.dropna()

In [ ]:
# create dataset
dataset = Dataset.from_pandas(df[["path","sentence"]])
dataset = dataset.cast_column("path", Audio(sampling_rate=16000))

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="ne", task="transcribe")

In [ ]:
def prepare(batch):
    audio = batch["path"]
    batch["input_features"] = processor(audio["array"], sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

In [ ]:
dataset = dataset.map(prepare)
dataset.set_format(type="torch", columns=["input_features","labels"])

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="ne", task="transcribe")

In [ ]:
# train/test split
train_test = dataset.train_test_split(test_size=0.1)

In [ ]:
def collate(batch):
    input_features = torch.stack([x["input_features"] for x in batch])
    labels = processor.tokenizer.pad({"input_ids":[x["labels"] for x in batch]}, return_tensors="pt").input_ids
    return {"input_features": input_features, "labels": labels}

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="./whisper-nepali",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=True,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="no"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_test["train"],
    eval_dataset=train_test["test"],
    tokenizer=processor.feature_extractor,
    data_collator=collate
)

In [ ]:
trainer.train()

In [ ]:
# demo inference
sample = dataset[0]
input_feat = sample["input_features"].unsqueeze(0)
pred_ids = model.generate(input_feat)
print("Prediction:", processor.tokenizer.decode(pred_ids[0], skip_special_tokens=True))
print("Target:", sample["sentence"])